# Domain-Adaptive Colorization — Pipeline Runner

Runs `main.py` end to end: Grounded-SAM detection/routing → per-domain DDColor experts → soft blending → RGB reconstruction.

**Kaggle settings:** GPU accelerator (T4 x2 or P100) and **Internet ON**.

In [ ]:
# Cell 1: clone the project (branch B)
from pathlib import Path

REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")
if not REPO_DIR.exists():
    !git clone -b B https://github.com/zaeem886/domain-adaptive-colorization.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Cell 2: install the Person A dependencies
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub pyyaml tqdm

In [ ]:
# Cell 3: clone DDColor and download the pretrained DDColor-T checkpoint
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

CHECKPOINT = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
print("DDColor:", DDCOLOR_DIR)
print("Checkpoint:", CHECKPOINT)

In [ ]:
# Cell 4: run the pipeline
IMAGE_PATH = "pics/dog.jpeg"          # change to your own image
OUTPUT_PATH = "outputs/kaggle_colorized.png"

!python main.py \
    --image {IMAGE_PATH} \
    --output {OUTPUT_PATH} \
    --single-checkpoint {CHECKPOINT} \
    --ddcolor-path {DDCOLOR_DIR} \
    --sam-checkpoint sam_vit_b.pth \
    --device cuda \
    --save-domain-map

In [ ]:
# Cell 5: display the input, result, and routed domain map
from IPython.display import Image as IPImage, display

print("Input")
display(IPImage(filename=IMAGE_PATH))
print("Colorized")
display(IPImage(filename=OUTPUT_PATH))
print("Domain map")
display(IPImage(filename=OUTPUT_PATH.replace(".png", "_domains.png")))

### Notes
- To skip detection and colorize the whole image with the general expert, add `--skip-detection` to the command above.
- For real per-domain experts, replace `--single-checkpoint` with `--checkpoints-dir checkpoints/`; the directory should contain `people.pth`, `nature.pth`, `urban.pth`, `animals.pth`, and `fallback.pth`.
- SAM ViT-B (~375 MB) is downloaded automatically on the first run if `sam_vit_b.pth` is missing.